<a href="https://colab.research.google.com/github/lilianaprj/Churn-prediction-project/blob/main/Churn_prediction_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
data_descriptions = pd.read_csv('data_descriptions.csv')
pd.set_option('display.max_colwidth', None)

print("=== TRAIN COLUMNS ===")
print(train_df.columns.tolist())
print("\n=== TRAIN SHAPE ===")
print(train_df.shape)
print("\n=== FIRST 3 ROWS ===")
print(train_df.head(3))
print("\n=== DATA DESCRIPTIONS ===")
print(data_descriptions)

=== TRAIN COLUMNS ===
['AccountAge', 'MonthlyCharges', 'TotalCharges', 'SubscriptionType', 'PaymentMethod', 'PaperlessBilling', 'ContentType', 'MultiDeviceAccess', 'DeviceRegistered', 'ViewingHoursPerWeek', 'AverageViewingDuration', 'ContentDownloadsPerMonth', 'GenrePreference', 'UserRating', 'SupportTicketsPerMonth', 'Gender', 'WatchlistSize', 'ParentalControl', 'SubtitlesEnabled', 'CustomerID', 'Churn']

=== TRAIN SHAPE ===
(243787, 21)

=== FIRST 3 ROWS ===
   AccountAge  MonthlyCharges  TotalCharges SubscriptionType PaymentMethod  \
0          20       11.055215    221.104302          Premium  Mailed check   
1          57        5.175208    294.986882            Basic   Credit card   
2          73       12.106657    883.785952            Basic  Mailed check   

  PaperlessBilling ContentType MultiDeviceAccess DeviceRegistered  \
0               No        Both                No           Mobile   
1              Yes      Movies                No           Tablet   
2              

In [10]:
print("=== Churn balance ===")
print(train_df['Churn'].value_counts())
print("Churn rate:", train_df['Churn'].mean().round(4))

print("\n=== Missing in train ===")
print(train_df.isnull().sum()[train_df.isnull().sum() > 0])
print("Total missing train:", train_df.isnull().sum().sum())

print("\n=== Missing in test ===")
print(test_df.isnull().sum()[test_df.isnull().sum() > 0])
print("Total missing test:", test_df.isnull().sum().sum())

cat_cols = train_df.select_dtypes(include='object').columns.drop('CustomerID')
print("\n=== Categorical cardinality ===")
for col in cat_cols:
    print(f"{col}: {train_df[col].nunique()} unique -> {train_df[col].unique()[:6]}")

print("\n=== TotalCharges sanity ===")
train_df['expected_charges'] = train_df['MonthlyCharges'] * train_df['AccountAge']
train_df['charge_diff'] = train_df['TotalCharges'] - train_df['expected_charges']
print(train_df[['MonthlyCharges','AccountAge','TotalCharges','expected_charges','charge_diff']].head())
print("Mean diff:", train_df['charge_diff'].mean().round(2))
print("Std diff:", np.round(train_df['charge_diff'].std(), 4))

=== Churn balance ===
Churn
0    199605
1     44182
Name: count, dtype: int64
Churn rate: 0.1812

=== Missing in train ===
Series([], dtype: int64)
Total missing train: 0

=== Missing in test ===
Series([], dtype: int64)
Total missing test: 0

=== Categorical cardinality ===
SubscriptionType: 3 unique -> ['Premium' 'Basic' 'Standard']
PaymentMethod: 4 unique -> ['Mailed check' 'Credit card' 'Electronic check' 'Bank transfer']
PaperlessBilling: 2 unique -> ['No' 'Yes']
ContentType: 3 unique -> ['Both' 'Movies' 'TV Shows']
MultiDeviceAccess: 2 unique -> ['No' 'Yes']
DeviceRegistered: 4 unique -> ['Mobile' 'Tablet' 'Computer' 'TV']
GenrePreference: 5 unique -> ['Sci-Fi' 'Action' 'Fantasy' 'Drama' 'Comedy']
Gender: 2 unique -> ['Male' 'Female']
ParentalControl: 2 unique -> ['No' 'Yes']
SubtitlesEnabled: 2 unique -> ['No' 'Yes']

=== TotalCharges sanity ===
   MonthlyCharges  AccountAge  TotalCharges  expected_charges   charge_diff
0       11.055215          20    221.104302        221.1043

In [11]:
for col in train_df.columns:
    if col in ('CustomerID', 'expected_charges', 'charge_diff'): continue
    if train_df[col].dtype == 'object':
        bad = train_df[col].isin(['Unknown', 'N/A', 'null', '', 'nan', 'None', '-']).sum()
        if bad > 0:
            print(f"{col}: {bad} sentinel values")
    else:
        bad = (train_df[col] == -1).sum() + (train_df[col] == -999).sum()
        if bad > 0:
            print(f"{col}: {bad} negative-sentinel values")

print("Sanity check done.")

Sanity check done.


In [12]:
!pip install lightgbm -q

In [13]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd

DIAG_COLS = ['expected_charges', 'charge_diff']

train_df = train_df.drop(columns=DIAG_COLS, errors='ignore')
test_df  = test_df.drop(columns=DIAG_COLS, errors='ignore')

X = train_df.drop(columns=['CustomerID', 'Churn']).copy()
y = train_df['Churn'].values
X_test = test_df.drop(columns=['CustomerID']).copy()

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

print("X shape:", X.shape, "| X_test shape:", X_test.shape)
print("Any object columns left?", X.dtypes.eq('object').sum())

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_data_in_leaf': 50,
    'verbose': -1,
    'seed': 42,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_aucs = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    dtrain = lgb.Dataset(X_tr, y_tr)
    dvalid = lgb.Dataset(X_va, y_va, reference=dtrain)

    model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=3000,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )

    oof_preds[va_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / skf.n_splits
    fold_aucs.append(roc_auc_score(y_va, oof_preds[va_idx]))
    print(f"Fold {fold+1}: AUC = {fold_aucs[-1]:.5f}  (best_iter={model.best_iteration})")

print(f"\nMean CV AUC: {np.mean(fold_aucs):.5f}")
print(f"OOF AUC (all folds combined): {roc_auc_score(y, oof_preds):.5f}")

X shape: (243787, 19) | X_test shape: (104480, 19)
Any object columns left? 0
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[156]	valid_0's auc: 0.746686
Fold 1: AUC = 0.74669  (best_iter=156)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[141]	valid_0's auc: 0.747952
Fold 2: AUC = 0.74795  (best_iter=141)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[121]	valid_0's auc: 0.746943
Fold 3: AUC = 0.74694  (best_iter=121)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[116]	valid_0's auc: 0.744755
Fold 4: AUC = 0.74476  (best_iter=116)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[118]	valid_0's auc: 0.747177
Fold 5: AUC = 0.74718  (best_iter=118)

Mean CV AUC: 0.74670
OOF AUC (all folds combined): 0.74670


In [14]:
print("Test preds min:", test_preds.min())
print("Test preds max:", test_preds.max())
print("Test preds mean:", test_preds.mean())
print("Train churn rate:", y.mean())
print()

prediction_df = pd.DataFrame({
    'CustomerID': test_df['CustomerID'],
    'predicted_probability': test_preds,
})

print("Shape:", prediction_df.shape)
print(prediction_df.head(10))
print(prediction_df['predicted_probability'].describe())

assert isinstance(prediction_df, pd.DataFrame), 'prediction_df must be a DataFrame'
assert prediction_df.columns[0] == 'CustomerID', 'First col must be CustomerID'
assert prediction_df.columns[1] == 'predicted_probability', 'Second col must be predicted_probability'
assert prediction_df.shape[0] == 104480, f'Must have 104480 rows, got {prediction_df.shape[0]}'
assert prediction_df.shape[1] == 2, f'Must have 2 cols, got {prediction_df.shape[1]}'
assert prediction_df['predicted_probability'].between(0, 1).all(), 'Probs must be in [0, 1]'
assert prediction_df['predicted_probability'].isnull().sum() == 0, 'No NaNs allowed'

print("\nAll format checks passed")

try:
    solution_df = pd.read_csv('test_sol.csv')
    grading_df = prediction_df.merge(solution_df, on='CustomerID', how='left')
    auc = roc_auc_score(y_true=grading_df['Churn'], y_score=grading_df['predicted_probability'])
    print(f"\nTEST AUC: {auc:.5f}")
except FileNotFoundError:
    print("\n(No test_sol.csv found — that's fine, you'll see the AUC after submitting.)")

Test preds min: 0.010331324600134448
Test preds max: 0.7913760189478484
Test preds mean: 0.18118496995579733
Train churn rate: 0.18123197709475894

Shape: (104480, 2)
   CustomerID  predicted_probability
0  O1W6BHP6RM               0.116735
1  LFR4X92X8H               0.046665
2  QM5GBIYODA               0.397862
3  D9RXTK2K9F               0.040609
4  ENTCCHR1LR               0.163919
5  7A88BB5IO6               0.391339
6  70OMW9XEWR               0.127704
7  EL1RMFMPYL               0.282460
8  4IA2QPT6ZK               0.219807
9  AEDCWHSJDN               0.184137
count    104480.000000
mean          0.181185
std           0.134802
min           0.010331
25%           0.077312
50%           0.142159
75%           0.250396
max           0.791376
Name: predicted_probability, dtype: float64

All format checks passed

TEST AUC: 0.74889


In [15]:
prediction_df.to_csv('submission_baseline.csv', index=False)

print("Saved submission_baseline.csv")
print("Shape:", prediction_df.shape)
print("Size: ~{:.1f} KB".format(prediction_df.memory_usage(deep=True).sum() / 1024))

print("\nFirst 5 rows:")
print(prediction_df.head())

Saved submission_baseline.csv
Shape: (104480, 2)
Size: ~6836.2 KB

First 5 rows:
   CustomerID  predicted_probability
0  O1W6BHP6RM               0.116735
1  LFR4X92X8H               0.046665
2  QM5GBIYODA               0.397862
3  D9RXTK2K9F               0.040609
4  ENTCCHR1LR               0.163919


In [16]:
def add_engagement_features(df):
    df = df.copy()
    df['WatchPerMonth']     = df['ViewingHoursPerWeek'] / (df['AccountAge'] + 1)
    df['DownloadsPerMonth'] = df['ContentDownloadsPerMonth'] / (df['AccountAge'] + 1)
    df['WatchlistPerMonth'] = df['WatchlistSize'] / (df['AccountAge'] + 1)
    df['SessionsPerWeek']   = df['ViewingHoursPerWeek'] / (df['AverageViewingDuration'] / 60 + 0.01)
    df['WatchToDownload']   = df['ViewingHoursPerWeek'] / (df['ContentDownloadsPerMonth'] + 1)
    df['HoursPerDollar']    = df['ViewingHoursPerWeek'] / (df['MonthlyCharges'] + 0.01)
    return df

train_fe = add_engagement_features(train_df)
test_fe  = add_engagement_features(test_df)

X = train_fe.drop(columns=['CustomerID', 'Churn']).copy()
y = train_fe['Churn'].values
X_test = test_fe.drop(columns=['CustomerID']).copy()

assert list(X.columns) == list(X_test.columns), "Column mismatch!"
print(f"X {X.shape}, X_test {X_test.shape}")

from sklearn.preprocessing import LabelEncoder

cat_cols_fe = X.select_dtypes(include='object').columns
for col in cat_cols_fe:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col]      = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

lgb_params = {
    'objective': 'binary', 'metric': 'auc',
    'learning_rate': 0.05, 'num_leaves': 63,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'bagging_freq': 5, 'min_data_in_leaf': 50,
    'verbose': -1, 'seed': 42,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_aucs  = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    dtrain = lgb.Dataset(X_tr, y_tr)
    dvalid = lgb.Dataset(X_va, y_va, reference=dtrain)

    model = lgb.train(
        lgb_params, dtrain,
        num_boost_round=3000,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )

    oof_preds[va_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / skf.n_splits
    fold_auc = roc_auc_score(y_va, oof_preds[va_idx])
    fold_aucs.append(fold_auc)
    print(f"Fold {fold+1}: AUC = {fold_auc:.5f}  (best_iter={model.best_iteration})")

print(f"\nMean CV AUC: {np.mean(fold_aucs):.5f}")
print(f"OOF AUC:     {roc_auc_score(y, oof_preds):.5f}")

baseline = 0.74670
print(f"\nBaseline: {baseline:.5f}")
print(f"Improvement: {np.mean(fold_aucs) - baseline:+.5f}")

X (243787, 25), X_test (104480, 25)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[106]	valid_0's auc: 0.746659
Fold 1: AUC = 0.74666  (best_iter=106)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[120]	valid_0's auc: 0.747764
Fold 2: AUC = 0.74776  (best_iter=120)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[114]	valid_0's auc: 0.746628
Fold 3: AUC = 0.74663  (best_iter=114)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[111]	valid_0's auc: 0.74425
Fold 4: AUC = 0.74425  (best_iter=111)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[123]	valid_0's auc: 0.747447
Fold 5: AUC = 0.74745  (best_iter=123)

Mean CV AUC: 0.74655
OOF AUC:     0.74653

Baseline: 0.74670
Improvement: -0.00015


In [17]:
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)

print(importance.head(25).to_string(index=False))

                 feature   importance
              AccountAge 63127.259829
          HoursPerDollar 44329.021245
  AverageViewingDuration 33670.892399
ContentDownloadsPerMonth 29659.835474
  SupportTicketsPerMonth 14982.314439
         SessionsPerWeek 10300.100083
       WatchlistPerMonth  9005.606127
          MonthlyCharges  8895.302124
     ViewingHoursPerWeek  8613.494935
       DownloadsPerMonth  5464.979159
         WatchToDownload  5078.369987
              UserRating  5032.879091
            TotalCharges  3521.590959
           WatchPerMonth  3438.993434
        SubscriptionType  2835.064790
           PaymentMethod  2600.663943
         GenrePreference  1762.585867
           WatchlistSize  1281.629628
             ContentType   736.040592
        DeviceRegistered   539.836931
        SubtitlesEnabled   419.312669
                  Gender   329.616313
       MultiDeviceAccess   206.616069
        PaperlessBilling   200.068719
         ParentalControl   198.708240


In [18]:
def add_features(df):
    df = df.copy()

    df['HoursPerDollar']    = df['ViewingHoursPerWeek'] / (df['MonthlyCharges'] + 0.01)
    df['SessionsPerWeek']   = df['ViewingHoursPerWeek'] / (df['AverageViewingDuration'] / 60 + 0.01)
    df['WatchlistPerMonth'] = df['WatchlistSize'] / (df['AccountAge'] + 1)

    df['TotalWeeklyWatch'] = df['AverageViewingDuration'] * df['SessionsPerWeek'] / 60

    df['EngagementScore'] = df['SessionsPerWeek'] * df['ContentDownloadsPerMonth']

    df['ShortSessions'] = (df['AverageViewingDuration'] < 30).astype(int)
    df['LongSessions']  = (df['AverageViewingDuration'] > 60).astype(int)

    df['WeeklyWatchRatio'] = df['ViewingHoursPerWeek'] / (7 * 24)

    df['AccountAgeSqrt'] = np.sqrt(df['AccountAge'])
    df['IsNewUser']      = (df['AccountAge'] < 6).astype(int)
    df['IsLongTerm']     = (df['AccountAge'] > 24).astype(int)

    return df

train_fe = add_features(train_df)
test_fe  = add_features(test_df)

print("Train shape:", train_fe.shape)
print("Test shape: ", test_fe.shape)
print("\nNew columns:")
print([c for c in train_fe.columns if c not in train_df.columns])

Train shape: (243787, 32)
Test shape:  (104480, 31)

New columns:
['HoursPerDollar', 'SessionsPerWeek', 'WatchlistPerMonth', 'TotalWeeklyWatch', 'EngagementScore', 'ShortSessions', 'LongSessions', 'WeeklyWatchRatio', 'AccountAgeSqrt', 'IsNewUser', 'IsLongTerm']


In [19]:
def add_engagement_features(df):
    df = df.copy()

    df['HoursPerDollar']    = df['ViewingHoursPerWeek'] / (df['MonthlyCharges'] + 0.01)
    df['SessionsPerWeek']   = df['ViewingHoursPerWeek'] / (df['AverageViewingDuration'] / 60 + 0.01)
    df['WatchlistPerMonth'] = df['WatchlistSize'] / (df['AccountAge'] + 1)

    df['TotalWeeklyWatch'] = df['AverageViewingDuration'] * df['SessionsPerWeek'] / 60

    df['EngagementScore'] = df['SessionsPerWeek'] * df['ContentDownloadsPerMonth']

    df['ShortSessions'] = (df['AverageViewingDuration'] < 30).astype(int)
    df['LongSessions']  = (df['AverageViewingDuration'] > 60).astype(int)

    df['WeeklyWatchRatio'] = df['ViewingHoursPerWeek'] / (7 * 24)

    df['AccountAgeSqrt'] = np.sqrt(df['AccountAge'])
    df['IsNewUser']      = (df['AccountAge'] < 6).astype(int)
    df['IsLongTerm']     = (df['AccountAge'] > 24).astype(int)
    return df

train_fe = add_engagement_features(train_df)
test_fe  = add_engagement_features(test_df)

X = train_fe.drop(columns=['CustomerID', 'Churn']).copy()
y = train_fe['Churn'].values
X_test = test_fe.drop(columns=['CustomerID']).copy()

assert list(X.columns) == list(X_test.columns), "Column mismatch!"
print(f"X {X.shape}, X_test {X_test.shape}")

from sklearn.preprocessing import LabelEncoder

cat_cols_fe = X.select_dtypes(include='object').columns
for col in cat_cols_fe:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col]      = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

lgb_params = {
    'objective': 'binary', 'metric': 'auc',
    'learning_rate': 0.05, 'num_leaves': 63,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'bagging_freq': 5, 'min_data_in_leaf': 50,
    'verbose': -1, 'seed': 42,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_aucs  = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    dtrain = lgb.Dataset(X_tr, y_tr)
    dvalid = lgb.Dataset(X_va, y_va, reference=dtrain)

    model = lgb.train(
        lgb_params, dtrain,
        num_boost_round=3000,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )

    oof_preds[va_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / skf.n_splits
    fold_auc = roc_auc_score(y_va, oof_preds[va_idx])
    fold_aucs.append(fold_auc)
    print(f"Fold {fold+1}: AUC = {fold_auc:.5f}  (best_iter={model.best_iteration})")

print(f"\nMean CV AUC: {np.mean(fold_aucs):.5f}")
print(f"OOF AUC:     {roc_auc_score(y, oof_preds):.5f}")

baseline = 0.74670
print(f"\nBaseline: {baseline:.5f}")
print(f"Improvement: {np.mean(fold_aucs) - baseline:+.5f}")

X (243787, 30), X_test (104480, 30)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[122]	valid_0's auc: 0.746607
Fold 1: AUC = 0.74661  (best_iter=122)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[129]	valid_0's auc: 0.748259
Fold 2: AUC = 0.74826  (best_iter=129)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[115]	valid_0's auc: 0.747204
Fold 3: AUC = 0.74720  (best_iter=115)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[114]	valid_0's auc: 0.744365
Fold 4: AUC = 0.74436  (best_iter=114)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[117]	valid_0's auc: 0.747125
Fold 5: AUC = 0.74712  (best_iter=117)

Mean CV AUC: 0.74671
OOF AUC:     0.74670

Baseline: 0.74670
Improvement: +0.00001


In [20]:
WEAK_FEATURES = ['Gender', 'MultiDeviceAccess', 'PaperlessBilling', 'ParentalControl']

train_fe = add_features(train_df).drop(columns=WEAK_FEATURES)
test_fe  = add_features(test_df).drop(columns=WEAK_FEATURES)

print("Train shape:", train_fe.shape)
print("Test shape: ", test_fe.shape)

X = train_fe.drop(columns=['CustomerID', 'Churn']).copy()
y = train_fe['Churn'].values
X_test = test_fe.drop(columns=['CustomerID']).copy()

assert list(X.columns) == list(X_test.columns)

from sklearn.preprocessing import LabelEncoder
cat_cols_fe = X.select_dtypes(include='object').columns
for col in cat_cols_fe:
    le = LabelEncoder()
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le.fit(combined)
    X[col]      = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))

import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

lgb_params = {
    'objective': 'binary', 'metric': 'auc',
    'learning_rate': 0.05, 'num_leaves': 63,
    'feature_fraction': 0.8, 'bagging_fraction': 0.8,
    'bagging_freq': 5, 'min_data_in_leaf': 50,
    'verbose': -1, 'seed': 42,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_aucs  = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]
    dtrain = lgb.Dataset(X_tr, y_tr)
    dvalid = lgb.Dataset(X_va, y_va, reference=dtrain)
    model = lgb.train(lgb_params, dtrain, num_boost_round=3000,
                     valid_sets=[dvalid],
                     callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    oof_preds[va_idx] = model.predict(X_va, num_iteration=model.best_iteration)
    test_preds += model.predict(X_test, num_iteration=model.best_iteration) / skf.n_splits
    fold_auc = roc_auc_score(y_va, oof_preds[va_idx])
    fold_aucs.append(fold_auc)
    print(f"Fold {fold+1}: AUC = {fold_auc:.5f}  (best_iter={model.best_iteration})")

print(f"\nMean CV AUC: {np.mean(fold_aucs):.5f}")
print(f"Baseline: 0.74670")
print(f"Improvement: {np.mean(fold_aucs) - 0.74670:+.5f}")

Train shape: (243787, 28)
Test shape:  (104480, 27)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[139]	valid_0's auc: 0.746779
Fold 1: AUC = 0.74678  (best_iter=139)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[94]	valid_0's auc: 0.747611
Fold 2: AUC = 0.74761  (best_iter=94)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[118]	valid_0's auc: 0.74704
Fold 3: AUC = 0.74704  (best_iter=118)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[138]	valid_0's auc: 0.744449
Fold 4: AUC = 0.74445  (best_iter=138)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[109]	valid_0's auc: 0.74744
Fold 5: AUC = 0.74744  (best_iter=109)

Mean CV AUC: 0.74666
Baseline: 0.74670
Improvement: -0.00004


In [21]:
!pip install optuna -q

In [24]:
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

X_tune = X.copy()
y_tune = y
X_test_tune = X_test.copy()

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 200),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'verbose': -1,
        'seed': 42,
    }

    skf_tune = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    aucs = []

    for tr_idx, va_idx in skf_tune.split(X_tune, y_tune):
        X_tr, X_va = X_tune.iloc[tr_idx], X_tune.iloc[va_idx]
        y_tr, y_va = y_tune[tr_idx], y_tune[va_idx]

        dtrain = lgb.Dataset(X_tr, y_tr)
        dvalid = lgb.Dataset(X_va, y_va, reference=dtrain)

        model_tune = lgb.train(
            params, dtrain,
            num_boost_round=2000,
            valid_sets=[dvalid],
            callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)],
        )

        preds = model_tune.predict(X_va, num_iteration=model_tune.best_iteration)
        aucs.append(roc_auc_score(y_va, preds))

    return np.mean(aucs)

study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=15, show_progress_bar=True)

print("\n=== Best params ===")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"\nBest 3-fold AUC: {study.best_value:.5f}")
print(f"Baseline 5-fold: 0.74670")

  0%|          | 0/15 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[162]	valid_0's auc: 0.747394
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[180]	valid_0's auc: 0.745877
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[173]	valid_0's auc: 0.744878
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[50]	valid_0's auc: 0.745747
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[53]	valid_0's auc: 0.744911
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[58]	valid_0's auc: 0.743477
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[473]	valid_0's auc: 0.747977
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[429]	valid_0's auc: 0.74

In [25]:
best_params = study.best_params.copy()
best_params.update({
    'objective': 'binary',
    'metric': 'auc',
    'verbose': -1,
    'seed': 42,
})

print("=== Best params (from Optuna) ===")
for k, v in best_params.items():
    print(f"  {k}: {v}")
print()

skf_fair = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_tuned = np.zeros(len(X_tune))
test_preds_tuned = np.zeros(len(X_test_tune))
fold_aucs_tuned = []

for fold, (tr_idx, va_idx) in enumerate(skf_fair.split(X_tune, y_tune)):
    X_tr, X_va = X_tune.iloc[tr_idx], X_tune.iloc[va_idx]
    y_tr, y_va = y_tune[tr_idx], y_tune[va_idx]

    dtrain = lgb.Dataset(X_tr, y_tr)
    dvalid = lgb.Dataset(X_va, y_va, reference=dtrain)

    m = lgb.train(
        best_params, dtrain,
        num_boost_round=3000,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )

    oof_preds_tuned[va_idx] = m.predict(X_va, num_iteration=m.best_iteration)
    test_preds_tuned += m.predict(X_test_tune, num_iteration=m.best_iteration) / 5
    fold_aucs_tuned.append(roc_auc_score(y_va, oof_preds_tuned[va_idx]))
    print(f"Fold {fold+1}: AUC = {fold_aucs_tuned[-1]:.5f}  (best_iter={m.best_iteration})")

print(f"\n=== Comparison ===")
print(f"Baseline 5-fold: 0.74670")
print(f"Tuned 5-fold:    {np.mean(fold_aucs_tuned):.5f}")
print(f"Improvement:     {np.mean(fold_aucs_tuned) - 0.74670:+.5f}")

=== Best params (from Optuna) ===
  learning_rate: 0.011546334401061205
  num_leaves: 15
  min_data_in_leaf: 64
  feature_fraction: 0.5034042725901844
  bagging_fraction: 0.6332979488105992
  bagging_freq: 3
  lambda_l2: 0.0470790252551951
  objective: binary
  metric: auc
  verbose: -1
  seed: 42

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1221]	valid_0's auc: 0.748005
Fold 1: AUC = 0.74800  (best_iter=1221)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1031]	valid_0's auc: 0.749514
Fold 2: AUC = 0.74951  (best_iter=1031)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1161]	valid_0's auc: 0.748875
Fold 3: AUC = 0.74888  (best_iter=1161)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1204]	valid_0's auc: 0.746385
Fold 4: AUC = 0.74638  (best_iter=1204)
Training until validation sc

In [26]:
study.optimize(objective, n_trials=15, show_progress_bar=True)

print(f"New best AUC: {study.best_value:.5f}")
print(f"New best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

  0%|          | 0/15 [00:00<?, ?it/s]

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[507]	valid_0's auc: 0.748383
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[367]	valid_0's auc: 0.747371
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[411]	valid_0's auc: 0.746012
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[794]	valid_0's auc: 0.748497
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[670]	valid_0's auc: 0.748155
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[651]	valid_0's auc: 0.746835
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1066]	valid_0's auc: 0.748666
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1108]	valid_0's auc:

In [27]:
best_params_final = study.best_params.copy()
best_params_final.update({
    'objective': 'binary',
    'metric': 'auc',
    'verbose': -1,
    'seed': 42,
})

print("=== Best params (from Optuna, 30 trials) ===")
for k, v in best_params_final.items():
    print(f"  {k}: {v}")
print()

skf_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds_final = np.zeros(len(X_tune))
test_preds_final = np.zeros(len(X_test_tune))
fold_aucs_final = []

for fold, (tr_idx, va_idx) in enumerate(skf_final.split(X_tune, y_tune)):
    X_tr, X_va = X_tune.iloc[tr_idx], X_tune.iloc[va_idx]
    y_tr, y_va = y_tune[tr_idx], y_tune[va_idx]
    dtrain = lgb.Dataset(X_tr, y_tr)
    dvalid = lgb.Dataset(X_va, y_va, reference=dtrain)
    m = lgb.train(best_params_final, dtrain, num_boost_round=3000,
                 valid_sets=[dvalid],
                 callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
    oof_preds_final[va_idx] = m.predict(X_va, num_iteration=m.best_iteration)
    test_preds_final += m.predict(X_test_tune, num_iteration=m.best_iteration) / 5
    fold_aucs_final.append(roc_auc_score(y_va, oof_preds_final[va_idx]))
    print(f"Fold {fold+1}: AUC = {fold_aucs_final[-1]:.5f}  (best_iter={m.best_iteration})")

print(f"\n=== Comparison ===")
print(f"Baseline:             0.74670")
print(f"Tuned (trial 14):     0.74838")
print(f"Tuned (trial 17):     {np.mean(fold_aucs_final):.5f}")
print(f"vs baseline:          {np.mean(fold_aucs_final) - 0.74670:+.5f}")

=== Best params (from Optuna, 30 trials) ===
  learning_rate: 0.0104974035830901
  num_leaves: 15
  min_data_in_leaf: 66
  feature_fraction: 0.5195900829956455
  bagging_fraction: 0.5838723583497234
  bagging_freq: 4
  lambda_l2: 3.3770145187015287e-05
  objective: binary
  metric: auc
  verbose: -1
  seed: 42

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1347]	valid_0's auc: 0.748084
Fold 1: AUC = 0.74808  (best_iter=1347)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1267]	valid_0's auc: 0.749657
Fold 2: AUC = 0.74966  (best_iter=1267)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[856]	valid_0's auc: 0.748353
Fold 3: AUC = 0.74835  (best_iter=856)
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1112]	valid_0's auc: 0.746467
Fold 4: AUC = 0.74647  (best_iter=1112)
Training until va

In [28]:
prediction_df = pd.DataFrame({
    'CustomerID': test_df['CustomerID'],
    'predicted_probability': test_preds_tuned, })

assert prediction_df.shape == (104480, 2)
assert list(prediction_df.columns) == ['CustomerID', 'predicted_probability']
assert prediction_df['predicted_probability'].between(0, 1).all()
assert prediction_df['predicted_probability'].isnull().sum() == 0

print("Format OK")
print(prediction_df.head())

prediction_df.to_csv('submission_final.csv', index=False)
print("\nSaved submission_final.csv")

try:
    solution_df = pd.read_csv('test_sol.csv')
    grading_df = prediction_df.merge(solution_df, on='CustomerID', how='left')
    auc = roc_auc_score(grading_df['Churn'], grading_df['predicted_probability'])
    print(f"\nTEST AUC: {auc:.5f}")
    print(f"   Baseline:  0.74889")
    print(f"   Improvement: {auc - 0.74889:+.5f}")
except FileNotFoundError:
    print("\n(No test_sol.csv — submit and you'll see the score)")

Format OK
   CustomerID  predicted_probability
0  O1W6BHP6RM               0.115587
1  LFR4X92X8H               0.047156
2  QM5GBIYODA               0.406373
3  D9RXTK2K9F               0.036517
4  ENTCCHR1LR               0.150184

Saved submission_final.csv

TEST AUC: 0.74954
   Baseline:  0.74889
   Improvement: +0.00065
